# ML-MODEL (HISTORICAL DATA)

## LOAD DATA

In [1]:
# import libraries
from pathlib import Path # paths
import pandas as pd # data-wrangling
import numpy as np # data-wrangling
import duckdb # databank
import seaborn as sns # plotting
import matplotlib.pyplot as plt # plotting

# import libraries for ml
from sklearn.model_selection import TimeSeriesSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, mean_absolute_error, mean_pinball_loss
import joblib


# set paths
PATH_BASE = Path.cwd().parent.parent
PATH_DATA = PATH_BASE / "data"
PATH_DB = PATH_DATA / "train.duckdb"
PATH_PARQUET = PATH_DATA / "train_delay_with_features.parquet"

# connect to db
con = duckdb.connect(str(PATH_DB))

con.execute(f"""
CREATE OR REPLACE TABLE train_delay AS
            SELECT * FROM
            read_parquet('{PATH_PARQUET}')
            """)


df_features = con.execute("SELECT * FROM train_delay").fetchdf()

## PREPROCESSING AND FEATURE SELECTION
Based on EDA in other script

In [2]:
# select features / split features and target
def choose_features_target(df):

    cols_exclude = ["ride_id", # id
                    "delay", # target
                    "departure_real","arrival_real", "departure_planned", "arrival_planned", # raw time variables
                    "train_name", "station_current", "station_start", "station_dest", # high-dimensional categories 
                    "hist_delay_train_q90", "hist_delay_station_q90", "stops_total", "stop_index", "share_ride_time" # discarded after correlation analysis
                    ]
    
    feature_cols = [col for col in df.columns if col not in cols_exclude]
    X = df[feature_cols]
    y = df["delay"]

    return X, y

# apply function
X, y = choose_features_target(df_features)

In [ ]:
# define features groups and their encoding for ml model
feature_scheme = {
    "categorical_one_hot": [
        "train_type", "station_role" # low number of dimensions (hot encoding ok)
    ],
    "numeric_scaled": [
        "temperature", "precipitation_log", "wind_speed", 
        "hour_sin", "hour_cos", "weekday_sin", "weekday_cos", 
        "month_sin", "month_cos", "travel_time", "dwell_time_planned", 
        "time_since_start_planned",
        "hist_delay_station_avg", "hist_delay_station_count",
        "hist_delay_train_avg", "hist_delay_train_count"
    ],
    "binary_passthrough": [
        "precipitation_any", "feast", "weekend", 
        "departure_rush_morning", "departure_rush_evening"
    ]
}


# define preprocessor 
preprocessor = ColumnTransformer(
    transformers=[
        # categories: one-hot 
        ("cat", OneHotEncoder(handle_unknown = "ignore", sparse_output = False), feature_scheme["categorical_one_hot"]),
        # numeric variables: standard scaling
        ("num", StandardScaler(), feature_scheme["numeric_scaled"]),
        # binary variables: do nothing
        ("pass", "passthrough", feature_scheme["binary_passthrough"])],
    # drop rest
    remainder = "drop")

## CROSS-VALIDATION
### DETERMINE GAP BETWEEN FOLDS

In [4]:
# get count of total observed hours and rows per hour (average)
total_hours = (df_features["departure_real"].max() - df_features["arrival_real"].min()).total_seconds() / 3600
rows_per_hour = len(df_features) / total_hours

# define number of hours that should be in between of folds
min_hours = 5

# calculate gap: rows per hour times gap
recommended_gap = int(rows_per_hour * min_hours)

print(f"Rows per hour: {rows_per_hour:.1f}")
print(f"Recommended minimal gap: {recommended_gap}")

Rows per hour: 139.8
Recommended minimal gap: 699


### DEFINE TIME SERIES CROSS-VALIDATOR

In [ ]:
ts_cv = TimeSeriesSplit(
    n_splits = 5,
    # lets be conservative: 
    # in very busy times, lower gap number might cause problems
    gap = 1000)

In [ ]:
# select data for training, tuning and testing

## tune on less data
X_tune = X.iloc[-600_000:-300_000]
y_tune = y.iloc[-600_000:-300_000]

## use almost all data for final training
X_train = X.iloc[:-300_000]
y_train = y.iloc[:-300_000]

## test on last observations
X_test = X.iloc[-300_000:]
y_test = y.iloc[-300_000:]

## TUNING THE MODEL

### PIPELINE

In [ ]:
# combines preprocessor and estimation method
pipe_hgb_mean_tune = make_pipeline(
    preprocessor,
    HistGradientBoostingRegressor(
        loss = "squared_error",
        random_state = 42,))

### HYPERPARAMETERS

In [ ]:
# define hyperparameterspace that should be tested
hyp_hgb = {
    "histgradientboostingregressor__learning_rate": loguniform(0.01, 0.2), 
    "histgradientboostingregressor__max_iter": randint(100, 500),        
    "histgradientboostingregressor__max_leaf_nodes": randint(15, 60),    
    "histgradientboostingregressor__min_samples_leaf": randint(20, 100),
}

### RANDOMIZEDSEARCH

In [ ]:
# initialize randomizedsearchcv
search_hgb = RandomizedSearchCV(
    pipe_hgb_mean_tune,
    param_distributions = hyp_hgb,
    n_iter = 15,               
    scoring = "neg_mean_absolute_error",
    cv = ts_cv,
    n_jobs = -1,
    verbose = 1,
    random_state = 42,)

In [ ]:
# search best parameter with tuning data
search_hgb.fit(X_tune, y_tune)

# get best parameters
best_hgb_params = search_hgb.best_params_

Fitting 5 folds for each of 15 candidates, totalling 75 fits


/Users/jakoberhard/venvs/TBA_project/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


385

## TRAINING

In [ ]:
# define the final pipelines (with best parameters)

## mean
pipe_hgb_mean = make_pipeline(
    preprocessor,
    HistGradientBoostingRegressor(
        loss = "squared_error",
        random_state = 42,)).set_params(**best_hgb_params)
## q05
pipe_hgb_q05 = make_pipeline(
    preprocessor,
    HistGradientBoostingRegressor(
        loss = "quantile",
        quantile = 0.05,
        random_state = 42,)).set_params(**best_hgb_params)
## q95
pipe_hgb_q95 = make_pipeline(
    preprocessor,
    HistGradientBoostingRegressor(
        loss = "quantile",
        quantile = 0.95,
        random_state = 42,)).set_params(**best_hgb_params)

# train the models
pipe_hgb_mean.fit(X_train, y_train)
pipe_hgb_q05.fit(X_train, y_train)
pipe_hgb_q95.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('histgradientboostingregressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the outpu

## TEST

In [ ]:
# function: evaluate the model
def evaluate_interval_model(model_mean, model_05, model_95, X_test, y_test):

    y_pred_50 = model_mean.predict(X_test)
    y_pred_05 = model_05.predict(X_test)
    y_pred_95 = model_95.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred_50)

    pinball_05 = mean_pinball_loss(y_test, y_pred_05, alpha = 0.05)
    pinball_95 = mean_pinball_loss(y_test, y_pred_95, alpha = 0.95)

    coverage = np.mean(
        (y_test >= y_pred_05) & (y_test <= y_pred_95))

    interval_width = np.mean(y_pred_95 - y_pred_05)

    return {
        "MAE": mae,
        "Pinball_05": pinball_05,
        "Pinball_95": pinball_95,
        "Coverage_90_interval": coverage,
        "Avg_Interval_Width": interval_width,
    }

In [ ]:
# apply the function
results_hgb = evaluate_interval_model(
    pipe_hgb_mean,
    pipe_hgb_q05,
    pipe_hgb_q95,
    X_test,
    y_test)

# print results
print(results_hgb)

## SAVE MODEL

In [ ]:
joblib.dump(pipe_hgb_mean, "src/jakob_analysis/pipeline_hgb_mean.pkl")
joblib.dump(pipe_hgb_q05, "src/jakob_analysis/pipeline_hgb_q05.pkl")
joblib.dump(pipe_hgb_q95, "src/jakob_analysis/pipeline_hgb_q95.pkl")

['src/jakob_analysis/pipeline_hgb_95.pkl']